# 🧠 Sistemas Experto
## Inferencia lógica y operadores AND / OR / NOT

## 1. Elementos de un sistema experto:

Un sistema experto trabaja con tres elementos fundamentales:

- **Hechos:** información que sabemos o que observamos.
- **Reglas:** relaciones que indican qué conclusión puede obtenerse cuando se cumplen determinadas condiciones.
- **Motor de inferencia:** mecanismo que analiza los hechos y aplica las reglas.

Podemos representarlo de forma sencilla:

**HECHOS + REGLAS → INFERENCIA → CONCLUSIÓN**

### Ejemplo

Supongamos que el sistema conoce:

- `humedad_alta`
- `lluvias_recientes`

Y posee la regla:

> SI hay humedad alta Y hubo lluvias recientes  
> ENTONCES revisar filtraciones.

Entonces puede obtener:

- `revisar_filtraciones`

Ese último dato **no estaba entre los hechos iniciales**. Es una conclusión obtenida mediante inferencia.

## 2. ¿Qué es inferir?

**Inferir es obtener una conclusión nueva a partir de información que ya conocemos.**

La idea central puede verse así:

```text
Hechos conocidos
      ↓
Evaluamos reglas
      ↓
¿Se cumplen sus condiciones?
      ↓
Sí → obtenemos una nueva conclusión
```

### Ejemplo manual

Tenemos:

- Hecho: `humedad_alta`
- Regla: SI `humedad_alta` → `revisar_impermeabilizacion`

Como la condición de la regla se cumple:

```text
humedad_alta
     ↓
R1
     ↓
revisar_impermeabilizacion
```

### 🧠 Pausa y piensa

Antes de programarlo:

**¿Qué debería pasar si NO tenemos `humedad_alta`?**

La regla no debería activarse, porque su condición no está satisfecha.

In [ ]:
# Representamos un conjunto de hechos conocidos.
hechos = {"humedad_alta"}

# Una regla muy simple: si está presente la condición,
# podemos obtener una nueva conclusión.
regla = {
    "si": {"humedad_alta"},
    "entonces": "revisar_impermeabilizacion"
}

if regla["si"].issubset(hechos):
    print("La regla se activa.")
    print("Nueva conclusión:", regla["entonces"])
else:
    print("La regla NO se activa.")

La regla se activa.
Nueva conclusión: revisar_impermeabilizacion


### 🔎 ¿Qué hizo el programa?

La parte importante es:

```python
regla["si"].issubset(hechos)
```

`regla["si"]` contiene las condiciones necesarias.

`hechos` contiene lo que actualmente conoce el sistema.

`issubset()` pregunta:

> **¿Todas las condiciones de la regla están dentro de los hechos conocidos?**

Si la respuesta es sí, la regla puede activarse.

## 3. Reglas con más de una condición

Una regla también puede necesitar **varios hechos al mismo tiempo**.

Por ejemplo:

> SI hay `humedad_alta` Y `lluvias_recientes`  
> ENTONCES `revisar_filtraciones`

### Ejemplo manual

Hechos:

```text
humedad_alta
lluvias_recientes
```

Condiciones de la regla:

```text
humedad_alta + lluvias_recientes
```

Como están presentes **las dos**, la regla se activa.

Pero si solamente tuviéramos:

```text
humedad_alta
```

la regla **no** podría activarse.

Esto nos prepara para entender el operador lógico **AND** que veremos más adelante.

In [ ]:
hechos = {"humedad_alta", "lluvias_recientes"}

regla = {
    "si": {"humedad_alta", "lluvias_recientes"},
    "entonces": "revisar_filtraciones"
}

print("Hechos conocidos:", hechos)
print("Condiciones necesarias:", regla["si"])

if regla["si"].issubset(hechos):
    print("✓ Se cumplen TODAS las condiciones.")
    print("→ Nueva conclusión:", regla["entonces"])
else:
    print("✗ Falta al menos una condición.")

Hechos conocidos: {'lluvias_recientes', 'humedad_alta'}
Condiciones necesarias: {'lluvias_recientes', 'humedad_alta'}
✓ Se cumplen TODAS las condiciones.
→ Nueva conclusión: revisar_filtraciones


## 4. Encadenamiento hacia adelante — Forward Chaining

El **Forward Chaining** comienza con los **hechos conocidos** y avanza hacia nuevas conclusiones.

La lógica es:

```text
HECHOS INICIALES
      ↓
EVALUAR REGLAS
      ↓
REGLA ACTIVADA
      ↓
NUEVO HECHO / CONCLUSIÓN
      ↓
VOLVER A EVALUAR REGLAS
```

Se continúa mientras aparezcan nuevas conclusiones.

### ¿Por qué se llama "hacia adelante"?

Porque partimos de lo que ya sabemos y avanzamos hacia lo que podemos deducir.

### Ejemplo manual de 3 pasos

Hecho inicial:

```text
lluvias_recientes
```

Reglas:

```text
R1: SI lluvias_recientes
    ENTONCES techo_humedo

R2: SI techo_humedo
    ENTONCES revisar_filtraciones

R3: SI revisar_filtraciones
    ENTONCES inspeccion_techo
```

El razonamiento sería:

```text
lluvias_recientes
       ↓ R1
techo_humedo
       ↓ R2
revisar_filtraciones
       ↓ R3
inspeccion_techo
```

### 🧠 Lo importante

La conclusión de una regla puede convertirse en un **nuevo hecho** y permitir que otra regla se active.

### 🔬 Hagamos primero el recorrido a mano

| Paso | Hechos disponibles | Regla que se activa | Nuevo hecho |
|---|---|---|---|
| 0 | `lluvias_recientes` | R1 | `techo_humedo` |
| 1 | `lluvias_recientes`, `techo_humedo` | R2 | `revisar_filtraciones` |
| 2 | + `revisar_filtraciones` | R3 | `inspeccion_techo` |
| 3 | + `inspeccion_techo` | ninguna nueva | fin |

Ahora vamos a pedirle a Python que haga este trabajo.

In [ ]:
reglas = [
    {
        "nombre": "R1",
        "si": {"lluvias_recientes"},
        "entonces": "techo_humedo"
    },
    {
        "nombre": "R2",
        "si": {"techo_humedo"},
        "entonces": "revisar_filtraciones"
    },
    {
        "nombre": "R3",
        "si": {"revisar_filtraciones"},
        "entonces": "inspeccion_techo"
    }
]

hechos = {"lluvias_recientes"}

for regla in reglas:
    if regla["si"].issubset(hechos):
        hechos.add(regla["entonces"])

print("Hechos obtenidos:", hechos)

Hechos obtenidos: {'inspeccion_techo', 'lluvias_recientes', 'revisar_filtraciones', 'techo_humedo'}


### ⚠️ Una observación importante

El código anterior sirve para visualizar la idea, pero tiene una limitación:

Si una regla produce un hecho que permite activar **otra regla anterior de la lista**, un único recorrido no necesariamente alcanza.

Por eso, un Forward Chaining completo suele repetir el proceso hasta que **ya no aparezcan hechos nuevos**.

In [ ]:
def encadenamiento_adelante(hechos_iniciales, reglas):
    hechos = set(hechos_iniciales)

    hubo_cambios = True

    while hubo_cambios:
        hubo_cambios = False

        for regla in reglas:
            condiciones = regla["si"]
            conclusion = regla["entonces"]

            if condiciones.issubset(hechos) and conclusion not in hechos:
                hechos.add(conclusion)
                hubo_cambios = True

    return hechos


hechos_iniciales = {"lluvias_recientes"}

resultado = encadenamiento_adelante(hechos_iniciales, reglas)

print("Hechos iniciales:", hechos_iniciales)
print("Hechos después de inferir:", resultado)

Hechos iniciales: {'lluvias_recientes'}
Hechos después de inferir: {'inspeccion_techo', 'lluvias_recientes', 'revisar_filtraciones', 'techo_humedo'}


### ¿Qué está haciendo el algoritmo?

1. Empieza con los hechos iniciales.
2. Recorre las reglas.
3. Pregunta si las condiciones de cada regla están presentes.
4. Si se cumplen, agrega la conclusión.
5. Vuelve a revisar las reglas.
6. Se detiene cuando ya no aparecen hechos nuevos.

### ¿Cuándo termina?

Cuando una vuelta completa no agrega ninguna conclusión nueva.

Ese punto significa:

> **No quedan reglas que puedan producir nuevos hechos a partir de lo que conocemos.**

## 5. 🏗️ Ejemplo aplicado: inspección de una obra

Ahora usamos un ejemplo más cercano al contexto de la clase.

### Hechos iniciales

- `humedad_alta`
- `lluvias_recientes`
- `fisuras_importantes`
- `edificio_antiguo`

### Reglas

- **R1:** humedad alta → revisar impermeabilización.
- **R2:** revisar impermeabilización + lluvias recientes → inspeccionar cubierta.
- **R3:** fisuras importantes → inspeccionar estructura.
- **R4:** inspeccionar estructura + edificio antiguo → solicitar evaluación especializada.

### 🧠 Antes de ejecutar

Podemos anticipar dos cadenas:

```text
humedad_alta
    ↓ R1
revisar_impermeabilizacion
    ↓ R2 + lluvias_recientes
inspeccionar_cubierta
```

y:

```text
fisuras_importantes
    ↓ R3
inspeccionar_estructura
    ↓ R4 + edificio_antiguo
solicitar_evaluacion_especializada
```

In [ ]:
reglas_obra = [
    {
        "nombre": "R1",
        "si": {"humedad_alta"},
        "entonces": "revisar_impermeabilizacion"
    },
    {
        "nombre": "R2",
        "si": {"revisar_impermeabilizacion", "lluvias_recientes"},
        "entonces": "inspeccionar_cubierta"
    },
    {
        "nombre": "R3",
        "si": {"fisuras_importantes"},
        "entonces": "inspeccionar_estructura"
    },
    {
        "nombre": "R4",
        "si": {"inspeccionar_estructura", "edificio_antiguo"},
        "entonces": "solicitar_evaluacion_especializada"
    }
]

hechos_obra = {
    "humedad_alta",
    "lluvias_recientes",
    "fisuras_importantes",
    "edificio_antiguo"
}

resultado_obra = encadenamiento_adelante(hechos_obra, reglas_obra)

print("Hechos iniciales:")
for hecho in sorted(hechos_obra):
    print(" -", hecho)

print("\nConclusiones / hechos obtenidos:")
for hecho in sorted(resultado_obra - hechos_obra):
    print(" -", hecho)

Hechos iniciales:
 - edificio_antiguo
 - fisuras_importantes
 - humedad_alta
 - lluvias_recientes

Conclusiones / hechos obtenidos:
 - inspeccionar_cubierta
 - inspeccionar_estructura
 - revisar_impermeabilizacion
 - solicitar_evaluacion_especializada


### 🔎 Interpretación

El sistema no "adivina" las conclusiones.

Cada conclusión aparece porque:

- existe un hecho necesario,
- existe una regla,
- las condiciones de esa regla se cumplen.

Por ejemplo:

```text
fisuras_importantes
       +
R3
       ↓
inspeccionar_estructura
```

Luego esa nueva conclusión puede participar en otra regla:

```text
inspeccionar_estructura
       +
edificio_antiguo
       +
R4
       ↓
solicitar_evaluacion_especializada
```

Eso es **encadenamiento**: una inferencia habilita otra inferencia.

## 6. Encadenamiento hacia atrás — Backward Chaining

El **Backward Chaining** funciona al revés.

En lugar de comenzar preguntando:

> "¿Qué puedo concluir con estos hechos?"

comienza preguntando:

> **"¿Puedo demostrar este objetivo?"**

### Ejemplo

Objetivo:

```text
solicitar_evaluacion_especializada
```

Buscamos una regla que permita obtenerlo:

```text
R4:
SI inspeccionar_estructura
Y edificio_antiguo
ENTONCES solicitar_evaluacion_especializada
```

Entonces el objetivo se transforma en dos subobjetivos:

```text
¿Está demostrado inspeccionar_estructura?
¿Está demostrado edificio_antiguo?
```

Para `inspeccionar_estructura` encontramos R3:

```text
R3:
SI fisuras_importantes
ENTONCES inspeccionar_estructura
```

Por lo tanto preguntamos:

```text
¿Tenemos fisuras_importantes?
```

Si la respuesta es sí, podemos demostrar `inspeccionar_estructura`.

Finalmente verificamos `edificio_antiguo`.

Si ambos están disponibles, podemos demostrar el objetivo.

### 🔬 Recorrido manual

```text
OBJETIVO
solicitar_evaluacion_especializada
          ↑
          │ R4
          │
   ┌──────┴────────┐
   ↑               ↑
inspeccionar_   edificio_antiguo
estructura
   ↑
   │ R3
   │
fisuras_importantes
```

Observá la diferencia:

- **Forward:** parte de `fisuras_importantes` y avanza.
- **Backward:** parte del objetivo y retrocede buscando qué necesita.

In [ ]:
def encadenamiento_atras(objetivo, hechos, reglas, camino=None):
    if camino is None:
        camino = set()

    # Si el objetivo ya es un hecho conocido, está demostrado.
    if objetivo in hechos:
        return True

    # Evita recorrer indefinidamente una cadena circular.
    if objetivo in camino:
        return False

    camino = camino | {objetivo}

    # Buscamos reglas que puedan producir el objetivo.
    reglas_candidatas = [
        regla for regla in reglas
        if regla["entonces"] == objetivo
    ]

    # Intentamos demostrar todas las condiciones de alguna regla.
    for regla in reglas_candidatas:
        condiciones_demostradas = all(
            encadenamiento_atras(condicion, hechos, reglas, camino)
            for condicion in regla["si"]
        )

        if condiciones_demostradas:
            return True

    return False


objetivo = "solicitar_evaluacion_especializada"

print(
    "¿Se puede demostrar el objetivo?",
    encadenamiento_atras(objetivo, hechos_obra, reglas_obra)
)

¿Se puede demostrar el objetivo? True


### 🧩 Entendiendo la función paso a paso

La función utiliza tres ideas importantes:

**1. Si el objetivo ya es un hecho, no hay que deducirlo.**

```python
if objetivo in hechos:
```

**2. Buscamos reglas cuya conclusión sea justamente el objetivo.**

```python
regla["entonces"] == objetivo
```

**3. Una regla con varias condiciones necesita que todas puedan demostrarse.**

Para eso usamos:

```python
all(...)
```

`all()` devuelve verdadero cuando **todas** las comprobaciones son verdaderas.

### ¿Y por qué aparece `camino`?

Sirve para evitar ciclos.

Por ejemplo:

```text
A → B
B → C
C → A
```

Sin una protección, el algoritmo podría continuar buscando indefinidamente.

## 7. 🚫 ¿Qué pasa cuando falta un hecho?

Supongamos que queremos demostrar:

```text
solicitar_evaluacion_especializada
```

pero eliminamos:

```text
fisuras_importantes
```

Entonces ya no podemos demostrar:

```text
inspeccionar_estructura
```

y, por lo tanto, tampoco podemos demostrar el objetivo final.

### Este ejemplo es importante

El Backward Chaining no dice simplemente "no".

Permite localizar **qué condición necesaria no pudo demostrarse**.

In [ ]:
hechos_sin_fisuras = {
    "humedad_alta",
    "lluvias_recientes",
    "edificio_antiguo"
}

objetivo = "solicitar_evaluacion_especializada"

resultado = encadenamiento_atras(
    objetivo,
    hechos_sin_fisuras,
    reglas_obra
)

print("Objetivo:", objetivo)
print("¿Se puede demostrar?", resultado)

Objetivo: solicitar_evaluacion_especializada
¿Se puede demostrar? False


## 8. 🔄 Forward vs. Backward

Ambas técnicas utilizan hechos y reglas, pero comienzan desde lugares diferentes.

| Característica | Forward Chaining | Backward Chaining |
|---|---|---|
| Punto de partida | Hechos conocidos | Objetivo |
| Dirección | Hechos → conclusiones | Objetivo → condiciones |
| Pregunta principal | ¿Qué puedo obtener? | ¿Puedo demostrar esto? |
| Uso conceptual | Explorar posibles conclusiones | Verificar una hipótesis |
| Puede generar nuevos hechos | Sí | Busca demostrar objetivos |

### Ejemplo simple

**Forward:**

```text
hechos → reglas → conclusiones
```

**Backward:**

```text
objetivo → reglas → condiciones → hechos
```

No se trata de memorizar solamente los nombres: lo importante es reconocer **desde dónde comienza el razonamiento**.

# 9. 🧠 Operadores lógicos: AND, OR y NOT

Ahora pasamos a la lógica que utilizamos dentro de las condiciones.

## AND

Significa **Y**.

Una condición con AND es verdadera solamente cuando **todas** sus partes son verdaderas.

Ejemplo:

> humedad alta **Y** fisuras importantes

Deben cumplirse ambas.

## OR

Significa **O**.

La condición es verdadera cuando **al menos una** de sus partes es verdadera.

Ejemplo:

> humedad alta **O** lluvias recientes

Alcanza con que se cumpla una.

## NOT

Significa **NO**.

Permite expresar que una condición no se cumple.

Ejemplo:

> **NO** hay señalización correcta.

### 📋 Tabla de verdad: AND

| A | B | A AND B |
|---|---|---|
| F | F | F |
| F | V | F |
| V | F | F |
| V | V | V |

### 📋 Tabla de verdad: OR

| A | B | A OR B |
|---|---|---|
| F | F | F |
| F | V | V |
| V | F | V |
| V | V | V |

### 📋 Tabla de verdad: NOT

| A | NOT A |
|---|---|
| F | V |
| V | F |

In [ ]:
# AND
humedad_alta = True
fisuras_importantes = True

print("AND:", humedad_alta and fisuras_importantes)

# OR
lluvias_recientes = False
print("OR:", humedad_alta or lluvias_recientes)

# NOT
senalizacion_correcta = False
print("NOT:", not senalizacion_correcta)

AND: True
OR: True
NOT: True


## 10. 🏗️ Operadores aplicados a una inspección

Podemos traducir reglas del dominio a expresiones de Python.

### Regla con AND

> Si la humedad es alta **Y** hay muchas fisuras → inspección estructural urgente.

```python
humedad > 70 and fisuras > 5
```

### Regla con OR

> Si la humedad es alta **O** hubo lluvia → revisar filtraciones.

```python
humedad > 70 or lluvia
```

### Regla con NOT

> Si la señalización **NO** es correcta → corregir señalización.

```python
not senalizacion_correcta
```

### 🧠 Observación

Los paréntesis ayudan a dejar claro cómo queremos agrupar las condiciones, especialmente cuando combinamos **AND** y **OR**.

In [ ]:
def evaluar_seguridad(
    humedad,
    fisuras,
    lluvia,
    senalizacion_correcta,
    usa_epp
):
    recomendaciones = []

    if humedad > 70 and fisuras > 5:
        recomendaciones.append("Inspección estructural urgente.")

    if humedad > 70 or lluvia:
        recomendaciones.append("Revisar posibles filtraciones.")

    if not senalizacion_correcta:
        recomendaciones.append("Corregir la señalización.")

    if not usa_epp:
        recomendaciones.append("Verificar uso de EPP.")

    if (humedad > 70 or lluvia) and fisuras > 3 and senalizacion_correcta:
        recomendaciones.append("Priorizar inspección técnica.")

    return recomendaciones


resultado = evaluar_seguridad(
    humedad=82,
    fisuras=7,
    lluvia=True,
    senalizacion_correcta=True,
    usa_epp=False
)

for recomendacion in resultado:
    print("-", recomendacion)

- Inspección estructural urgente.
- Revisar posibles filtraciones.
- Verificar uso de EPP.
- Priorizar inspección técnica.


### 🔎 ¿Cómo interpretar el resultado?

No debemos mirar solamente las recomendaciones finales. Conviene preguntarnos **qué regla produjo cada una**.

Por ejemplo, si:

```text
humedad = 82
fisuras = 7
```

entonces:

```python
humedad > 70
```

es verdadero y:

```python
fisuras > 5
```

también es verdadero.

Por eso:

```python
humedad > 70 and fisuras > 5
```

resulta verdadero.

En cambio, si:

```text
usa_epp = False
```

entonces:

```python
not usa_epp
```

es verdadero y se activa la recomendación correspondiente.

## 11. 🔀 Combinando AND y OR

Una regla puede tener una estructura más compleja.

Por ejemplo:

> Si hay humedad alta **O** lluvia,  
> **Y además** hay más de 3 fisuras,  
> **Y** la señalización es correcta,  
> entonces priorizar una inspección técnica.

En Python:

```python
(humedad > 70 or lluvia) and fisuras > 3 and senalizacion_correcta
```

### Paso a paso

Primero evaluamos:

```python
(humedad > 70 or lluvia)
```

Después:

```python
fisuras > 3
```

Y finalmente:

```python
senalizacion_correcta
```

Las tres partes deben resultar verdaderas porque están unidas mediante `and`.

### 🧠 Idea clave

Los paréntesis hacen visible qué parte queremos evaluar primero y ayudan a evitar interpretaciones equivocadas.

In [ ]:
# Probemos tres situaciones diferentes.

escenarios = [
    {
        "nombre": "Escenario 1",
        "humedad": 80,
        "fisuras": 5,
        "lluvia": False,
        "senalizacion_correcta": True,
        "usa_epp": True
    },
    {
        "nombre": "Escenario 2",
        "humedad": 60,
        "fisuras": 2,
        "lluvia": True,
        "senalizacion_correcta": True,
        "usa_epp": True
    },
    {
        "nombre": "Escenario 3",
        "humedad": 60,
        "fisuras": 2,
        "lluvia": False,
        "senalizacion_correcta": False,
        "usa_epp": False
    }
]

for escenario in escenarios:
    print("\n", escenario["nombre"])

    recomendaciones = evaluar_seguridad(
        escenario["humedad"],
        escenario["fisuras"],
        escenario["lluvia"],
        escenario["senalizacion_correcta"],
        escenario["usa_epp"]
    )

    if recomendaciones:
        for r in recomendaciones:
            print("-", r)
    else:
        print("- No se activaron recomendaciones.")


 Escenario 1
- Revisar posibles filtraciones.
- Priorizar inspección técnica.

 Escenario 2
- Revisar posibles filtraciones.

 Escenario 3
- Corregir la señalización.
- Verificar uso de EPP.


# 12. 🧠 Ejemplo integrador

Ahora reunimos las ideas principales.

Tenemos:

### Hechos

```text
humedad_alta
lluvias_recientes
fisuras_importantes
edificio_antiguo
```

### Reglas

```text
R1: humedad_alta
    → revisar_impermeabilizacion

R2: revisar_impermeabilizacion + lluvias_recientes
    → inspeccionar_cubierta

R3: fisuras_importantes
    → inspeccionar_estructura

R4: inspeccionar_estructura + edificio_antiguo
    → solicitar_evaluacion_especializada
```

### Forward Chaining

Comenzamos con los hechos y avanzamos:

```text
hechos iniciales
      ↓
R1 / R3
      ↓
nuevos hechos
      ↓
R2 / R4
      ↓
nuevas conclusiones
```

### Backward Chaining

Si el objetivo fuera:

```text
solicitar_evaluacion_especializada
```

comenzamos por él y buscamos qué necesitamos demostrar:

```text
solicitar_evaluacion_especializada
             ↑
      inspeccionar_estructura
             ↑
      fisuras_importantes
```

y también:

```text
edificio_antiguo
```

Si todas las condiciones necesarias están disponibles, el objetivo puede demostrarse.

# 13. Cómo leer un sistema experto

Cuando encuentres un sistema experto basado en reglas, hacete estas preguntas:

### 1️⃣ ¿Qué hechos conoce inicialmente?

Son los datos de entrada.

### 2️⃣ ¿Qué reglas existen?

Cada regla indica qué condiciones permiten obtener una conclusión.

### 3️⃣ ¿Qué estrategia de inferencia se está utilizando?

- **Forward:** empieza con hechos.
- **Backward:** empieza con un objetivo.

### 4️⃣ ¿Qué condiciones deben cumplirse?

Acá aparecen operadores como:

- `AND`
- `OR`
- `NOT`

### 5️⃣ ¿Qué nueva información aparece?

Una conclusión puede convertirse en un nuevo hecho y activar otra regla.

---

## 🎯 Idea central de la clase

Un sistema experto no obtiene conclusiones de manera arbitraria.

**Razona aplicando reglas sobre hechos disponibles.**

Y podemos orientar ese razonamiento de dos maneras:

```text
FORWARD
Hechos → Reglas → Conclusiones
```

```text
BACKWARD
Objetivo → Reglas → Condiciones → Hechos
```